Import Python modules

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import glob

Read in counts data

In [2]:
counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)
counts_df = counts_df.replace('', np.nan)
counts_df.head()

/tmp/ipykernel_2694933/3899154490.py:1: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  counts_df = pd.read_csv('../results/counts.csv', keep_default_na=False)


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,branch_length,mut_class,mut_type,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate
0,689,A689C,A,C,HA,2,230,GAG,GCG,E,...,891,nonsynonymous,AC,H1,HA,HA_H1,1700,avian,0.524118,0.0
1,1402,A1402C,A,C,HA,1,468,AAT,CAT,N,...,9,nonsynonymous,AC,H3,HA,HA_H3,1700,all,0.005294,0.0
2,1402,A1402C,A,C,HA,1,468,AAG,CAG,K,...,20,nonsynonymous,AC,H3,HA,HA_H3,1700,all,0.011765,0.0
3,1402,A1402C,A,C,HA,1,468,AAG,CAG,K,...,43,nonsynonymous,AC,H3,HA,HA_H3,1700,all,0.025294,0.0
4,1402,A1402C,A,C,HA,1,468,AAG,CAG,K,...,4,nonsynonymous,AC,H3,HA,HA_H3,1700,all,0.002353,0.0


Read in predicted rates from neutral model

In [3]:
predicted_rates_df = pd.read_csv(
    '../results/expected_rates.csv',
    keep_default_na=False
)
del predicted_rates_df['tau_squared']
predicted_rates_df.head()

,mut_type,segment,motif,predicted_rate
0,AC,HA,AAA,0.132981
1,AC,HA,AAC,0.180268
2,AC,HA,AAG,0.156194
3,AC,HA,AAT,0.195275
4,AC,HA,CAA,0.113872


Compute expected counts and subset the data to mutations with at least X actual or expected counts

In [4]:
counts_cutoff = 10
actual_expected_df = (
    counts_df
    .merge(predicted_rates_df, on=['mut_type', 'segment', 'motif'], how='left')
    .assign(expected_count=lambda x: x['predicted_rate'] * x['evo_opp'])
    # subset to rows with at least X actual or expected counts
    .query("actual_count >= @counts_cutoff | expected_count >= @counts_cutoff")
)
actual_expected_df.to_csv('../results/actual_expected.csv', index=False)
print("Number of unique nucleotide mutations with data:", len(actual_expected_df))
actual_expected_df.head()

Number of unique nucleotide mutations with data: 129187


,site,nt_mut,wt_nt,mut_nt,gene,codon_position,codon_site,wt_codon,mut_codon,wt_aa,...,mut_type,subtype,segment,segment_subtype,segment_length,host,evo_opp,rate,predicted_rate,expected_count
38,1442,A1442C,A,C,PB1,2,481,AAG,ACG,K,...,AC,all,PB1,PB1_all,2273,all,83.072151,0.012038,0.199218,16.549457
66,1399,A1399C,A,C,HA,1,467,ACA,CCA,T,...,AC,H3,HA,HA_H3,1700,all,75.115294,0.000000,0.180268,13.540909
71,1453,A1453C,A,C,PB1,1,485,AAT,CAT,N,...,AC,all,PB1,PB1_all,2273,all,70.265288,0.000000,0.169610,11.917730
79,1454,A1454C,A,C,PB1,2,485,AAT,ACT,N,...,AC,all,PB1,PB1_all,2273,all,70.344039,0.000000,0.249063,17.520131
136,1423,A1423C,A,C,PB1,1,475,ATC,CTC,I,...,AC,all,PB1,PB1_all,2273,all,67.749670,0.000000,0.249063,16.873969


Compute fitness effects of synonymous nucleotide mutations, aggregating data for mutations away from the same wildtype nucleotide at the same site

In [5]:
pseudo_count = 0.5
groupby_cols = [
    'subtype', 'segment', 'gene', 'site', 'codon_site', 'wt_nt', 'mut_class'
]
syn_nt_fitness_df = (
    actual_expected_df
    .query("mut_class == 'synonymous'")
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
syn_nt_fitness_df.to_csv('../results/sitewise_synonymous_fitness_effects.csv', index=False)
print("Number of unique sites with estimated synonymous fitness effects:", len(syn_nt_fitness_df))
syn_nt_fitness_df.head()

Number of unique sites with estimated synonymous fitness effects: 11370


,subtype,segment,gene,site,codon_site,wt_nt,mut_class,actual_count,expected_count,delta_fitness
0,H1,HA,HA,6,2,A,synonymous,11,6.636002,0.477194
1,H1,HA,HA,6,2,G,synonymous,248,328.544137,-0.280749
2,H1,HA,HA,9,3,A,synonymous,37,22.921977,0.470666
3,H1,HA,HA,9,3,A,synonymous,31,93.128920,-1.089352
4,H1,HA,HA,13,5,C,synonymous,157,254.868422,-0.483282


Compute fitness effects of amino-acid mutations, aggregating data across nucleotide mutations that result in the same amino-acid mutation.

In [6]:
groupby_cols = [
    'subtype', 'segment', 'gene', 'codon_site', 'wt_aa', 'mut_aa', 'aa_mut',
    'mut_class'
]
aa_fitness_df = (
    actual_expected_df
    .groupby(groupby_cols, as_index=False)
    .agg({'actual_count': 'sum', 'expected_count': 'sum'})
    .assign(
        delta_fitness=lambda x: \
            np.log((x['actual_count'] + pseudo_count) / (x['expected_count'] + pseudo_count))
    )
)
aa_fitness_df.to_csv('../results/aa_fitness_effects.csv', index=False)
print("Number of unique aa muts with estimated fitness effects:", len(aa_fitness_df))
aa_fitness_df.head()

Number of unique aa muts with estimated fitness effects: 40910


,subtype,segment,gene,codon_site,wt_aa,mut_aa,aa_mut,mut_class,actual_count,expected_count,delta_fitness
0,H1,HA,HA,2,K,M,K2M,nonsynonymous,14,32.825608,-0.832177
1,H1,HA,HA,3,A,A,A3A,synonymous,37,22.921977,0.470666
2,H1,HA,HA,4,I,L,I4L,nonsynonymous,6,25.004579,-1.367056
3,H1,HA,HA,5,L,L,L5L,synonymous,9,26.267859,-1.035910
4,H1,HA,HA,6,V,V,V6V,synonymous,16,26.010929,-0.474197


Report the number of amino-acid level mutations with data in each category (including "synonymous" mutations that don't change the amino acid)

In [7]:
aa_fitness_df.groupby('mut_class').size()

mut_class
nonsense          2245
nonsynonymous    31379
synonymous        7286
dtype: int64

In [8]:
aa_fitness_df[aa_fitness_df['mut_class'] == 'nonsynonymous'].groupby(['segment', 'subtype']).size()

segment  subtype
HA       H1         2789
         H3         2728
         H5         1318
         H7           15
         H9          757
MP       all        1829
NA       N1         2185
         N2         2264
         N6           94
         N8           90
         N9            8
NP       all        3285
NS       all        1883
PA       all        4203
PB1      all        4052
PB2      all        3879
dtype: int64